In [41]:
import pandas as pd
import pyarrow.parquet as pq
from tqdm.auto import tqdm

In [42]:
path = 'Datasets/'
CHUNK_SIZE = 500_000

In [43]:
pf = pq.ParquetFile(path + "SharedResponses_clean.parquet")
total_rows = pf.metadata.num_rows
total_chunks = (total_rows // CHUNK_SIZE) + 1
print(f"Total rows: {total_rows:,} → {total_chunks} chunks of {CHUNK_SIZE:,}")

Total rows: 62,231,803 → 125 chunks of 500,000


In [44]:
survey = (
    pd.read_csv(path + "SharedResponsesSurvey.csv", low_memory=False,
                usecols=['ExtendedSessionID', 'Review_age', 'Review_gender', 'Review_education',
                        'Review_income', 'Review_political', 'Review_religious'])
    .drop_duplicates(subset='ExtendedSessionID')
)
print(f"Survey sessions loaded: {len(survey):,}")

Survey sessions loaded: 492,845


In [45]:
chunks = []
pbar = tqdm(pf.iter_batches(batch_size=CHUNK_SIZE), total=total_chunks, desc="Loading chunks")
try:
    for batch in pbar:
        chunk = batch.to_pandas()
        if isinstance(chunk['ExtendedSessionID'].dtype, pd.CategoricalDtype):
            chunk['ExtendedSessionID'] = chunk['ExtendedSessionID'].astype(str)
        chunk = chunk.merge(survey, on='ExtendedSessionID', how='left')
        chunks.append(chunk)
finally:
    pbar.close()

print("Concatenating chunks...", end=" ", flush=True)
df = pd.concat(chunks, ignore_index=True)
print("done.")

print(f"\nFinal shape: {df.shape}")
print(f"Rows with survey data: {df['Review_gender'].notna().sum():,} ({df['Review_gender'].notna().mean()*100:.1f}%)")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
print(df.head())

Loading chunks: 265it [01:21,  3.27it/s]                         

Concatenating chunks... 

done.

Final shape: (62231803, 40)
Rows with survey data: 10,046,584 (16.1%)
Memory usage: 8.01 GB
               ExtendedSessionID  Intervention  PedPed  Barrier  \
0    32757157_6999801415950060.0         False   False    False   
1  -1613944085_422160228641876.0         False    True    False   
2   1425316635_327833569077076.0         False   False     True   
3  -1683127088_785070916172117.0         False    True    False   
4  1525185249_1436495773909467.0         False   False     True   

   CrossingSignal AttributeLevel ScenarioTypeStrict DefaultChoice  \
0               1            Fit            Fitness           Fit   
1               1         Female             Gender          Male   
2               0            Old                Age         Young   
3               2           More        Utilitarian          More   
4               0            Low      Social Status          High   

  NonDefaultChoice  DefaultChoiceIsOmission  ...  FemaleDoctor  MaleDoctor  \
0    

In [47]:
df = df.dropna(axis=0)
df

,ExtendedSessionID,Intervention,PedPed,Barrier,CrossingSignal,AttributeLevel,ScenarioTypeStrict,DefaultChoice,NonDefaultChoice,DefaultChoiceIsOmission,...,FemaleDoctor,MaleDoctor,Dog,Cat,Review_age,Review_education,Review_gender,Review_income,Review_political,Review_religious
4,1525185249_1436495773909467.0,False,False,True,0,Low,Social Status,High,Low,False,...,0,0,0,0,26.0,bachelor,male,5000,0.50,0.50
17,-1033736141_5392791780749771.0,False,False,True,0,Male,Gender,Male,Female,True,...,0,0,0,0,14.0,underHigh,female,default,0.50,0.57
31,-972264246_4277296232223713.0,False,False,True,0,Pets,Species,Hoomans,Pets,False,...,0,0,4,1,20.0,high,male,under5000,0.75,0.84
34,-841718081_3084184331213722.0,False,False,False,2,Pets,Species,Hoomans,Pets,False,...,0,0,2,2,19.0,high,male,default,1.00,0.00
40,1247311683_3858850116393528.0,False,True,False,2,Female,Gender,Male,Female,False,...,2,0,0,0,38.0,others,male,above100000,0.68,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62231755,725752586_6904186263183265.0,True,False,False,0,Pets,Species,Hoomans,Pets,True,...,0,0,4,1,22.0,high,male,35000,0.50,0.00
62231759,1479950156_1327764658273678.0,True,True,False,2,More,Utilitarian,More,Less,False,...,2,0,0,0,40.0,college,male,5000,0.76,0.17
62231764,-1785551894_9617570104574178.0,True,False,False,0,Less,Utilitarian,More,Less,True,...,0,0,0,0,16.0,underHigh,male,default,0.78,0.00
62231773,-671133433_3786068032082825.0,True,True,False,1,Less,Utilitarian,More,Less,True,...,0,0,0,0,20.0,college,male,under5000,0.14,0.19
